# 11.3 Understanding Time-Based Data

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 11: Time Series Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-11-time-series-machine-learning/11.03_understanding-time-based-data.ipynb)

Companion code for Section 11.3 of the book. Run the cells in order from top to bottom.

**Example 11.3.1**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Creating DateTime-indexed Series ─────────────────────────────
np.random.seed(42)

# Various frequency options
print('=== Pandas DateTime Frequency Aliases ===')
freq_guide = {
    'D':  'Calendar day', 'B':   'Business day', 'H':  'Hourly',
    'T':  'Minutely',     'S':   'Secondly',      'W':  'Weekly',
    'M':  'Month end',    'MS':  'Month start',   'Q':  'Quarter end',
    'QS': 'Quarter start','A':   'Year end',      'AS': 'Year start',
}
for alias, meaning in freq_guide.items():
    print(f'  {alias:<4}: {meaning}')

# Create hourly data for 1 year
idx = pd.date_range('2023-01-01', periods=8760, freq='H')
values = (
    100
    + 20 * np.sin(2*np.pi*np.arange(8760)/24)       # Daily cycle
    + 15 * np.sin(2*np.pi*np.arange(8760)/(24*7))   # Weekly cycle
    + 10 * np.arange(8760)/8760                       # Linear trend
    + np.random.normal(0, 5, 8760)                    # Noise
)
ts_hourly = pd.Series(values, index=idx, name='energy_kwh')

# ── Resampling ────────────────────────────────────────────────────
# Aggregate to different frequencies
ts_daily  = ts_hourly.resample('D').agg({'sum','mean','max','min'})
ts_weekly = ts_hourly.resample('W').sum()
ts_monthly= ts_hourly.resample('M').sum()

print(f'\nHourly: {len(ts_hourly):,} rows → Daily: {len(ts_daily)} rows → Monthly: {len(ts_monthly)} rows')

# ── DateTime Indexing and Selection ──────────────────────────────
# All data from a specific year
ts_2023 = ts_hourly['2023']

# Specific month
ts_jan  = ts_hourly['2023-01']

# Date range
ts_range= ts_hourly['2023-03-01':'2023-03-31']

# Specific time of day (9 AM across all days)
ts_9am  = ts_hourly[ts_hourly.index.hour == 9]

print(f'\nDateTime selection examples:')
print(f'  Full year:        {len(ts_2023)} hours')
print(f'  January only:     {len(ts_jan)} hours')
print(f'  March only:       {len(ts_range)} hours')
print(f'  9 AM readings:    {len(ts_9am)} observations')

# ── Rolling Windows ───────────────────────────────────────────────
ts_daily_mean = ts_hourly.resample('D').mean()

ts_daily_mean_df = pd.DataFrame({'value': ts_daily_mean})
ts_daily_mean_df['rolling_7d']  = ts_daily_mean.rolling(7).mean()
ts_daily_mean_df['rolling_30d'] = ts_daily_mean.rolling(30).mean()
ts_daily_mean_df['ewm_7d']      = ts_daily_mean.ewm(span=7).mean()
ts_daily_mean_df['rolling_std'] = ts_daily_mean.rolling(7).std()

print('\nRolling statistics (last 5 days):')
print(ts_daily_mean_df.tail(5).round(3))

# ── Handling missing timestamps ───────────────────────────────────
# Introduce missing values
ts_missing = ts_daily_mean.copy()
missing_idx = np.random.choice(len(ts_missing), 20, replace=False)
ts_missing.iloc[missing_idx] = np.nan

print(f'\nMissing values: {ts_missing.isna().sum()}')

# Fill strategies
ts_ffill   = ts_missing.ffill()          # Forward fill
ts_bfill   = ts_missing.bfill()          # Backward fill
ts_linear  = ts_missing.interpolate('linear')     # Linear interpolation
ts_time    = ts_missing.interpolate('time')        # Time-weighted interpolation
ts_spline  = ts_missing.interpolate('spline', order=3)  # Cubic spline

# Compute error on known missing values
true_vals  = ts_daily_mean.iloc[missing_idx]
for name, filled in [('Forward Fill',  ts_ffill),
                      ('Linear Interp', ts_linear),
                      ('Time Interp',   ts_time),
                      ('Spline',        ts_spline)]:
    error = np.abs(filled.iloc[missing_idx] - true_vals).mean()
    print(f'  {name:<16}: MAE = {error:.4f}')

# ── Visualise ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Time Series Data Operations', fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(ts_daily_mean_df.index, ts_daily_mean_df['value'], alpha=0.4, color='gray', lw=1, label='Daily')
ax.plot(ts_daily_mean_df.index, ts_daily_mean_df['rolling_7d'],  color='#1976D2', lw=2, label='7-day MA')
ax.plot(ts_daily_mean_df.index, ts_daily_mean_df['rolling_30d'], color='#E53935', lw=2, label='30-day MA')
ax.plot(ts_daily_mean_df.index, ts_daily_mean_df['ewm_7d'],      color='#43A047', lw=2, ls='--', label='EWM(span=7)')
ax.set_title('Original + Rolling Windows'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.fill_between(ts_daily_mean_df.index,
               ts_daily_mean_df['rolling_7d'] - ts_daily_mean_df['rolling_std'],
               ts_daily_mean_df['rolling_7d'] + ts_daily_mean_df['rolling_std'],
               alpha=0.3, color='#1976D2', label='±1 std band')
ax.plot(ts_daily_mean_df.index, ts_daily_mean_df['rolling_7d'], color='#1976D2', lw=2, label='7-day MA')
ax.set_title('Rolling Mean ± Rolling Std (Bollinger Band style)'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('ts_operations.png', dpi=150, bbox_inches='tight')
plt.show()